# Machine Learning in Python - Project 

Due Friday, Apr 10th by 4 pm.

*Include contributors names here*

## Setup

*Install any packages here, define any functions if neeed, and load data*

In [36]:
# Add any additional libraries or submodules below

# Data libraries
import pandas as pd
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn modules
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from scipy.stats import chi2_contingency
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)


In [10]:
# Load data  
df_raw = pd.read_csv("unicef_malawi.csv")
df.head()

,HH1,HH2,LN,FS4,CB3,CB4,CB5A,CB5B,CB7,CB11,...,WS3,WS4,WS7,WS11,WS14,WS15,HW5,MA2_status,CL3_status,CL13_status
0,1.0,2.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,ELSEWHERE,5.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,ELSEWHERE,YES,NO,observed,observed,observed
1,1.0,3.0,1.0,1.0,5.0,YES,ECE,NaN,YES,NO,...,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITHOUT SLAB / OPEN PIT,IN OWN YARD / PLOT,NO,YES,structural_missing,structural_missing,structural_missing
2,1.0,4.0,2.0,2.0,16.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,ELSEWHERE,6.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,YES,observed,observed,observed
3,1.0,8.0,2.0,2.0,13.0,YES,PRIMARY,CLASS/GRADE 7,YES,NO,...,ELSEWHERE,30.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO,observed,observed,observed
4,1.0,10.0,2.0,2.0,14.0,YES,PRIMARY,CLASS/GRADE 6,YES,NO,...,ELSEWHERE,8.0,"NO, ALWAYS SUFFICIENT",PIT LATRINE: PIT LATRINE WITH SLAB,IN OWN YARD / PLOT,NO,NO,observed,observed,observed


# Introduction

*This section should include a brief introduction to the task and the data (assume this is a report you are delivering to a professional body (e.g. UNICEF)).*

*Briefly outline the approaches being used and the conclusions that you are able to draw.*

### 1.1 Task Overview and Objectives
This study aims to utilize high-dimensional survey data to identify and quantify the core socioeconomic and family determinants of **Childhood depression** in Malawi in order to develop a classification and predictive model. This report presents a robust feature selection and predictive modeling framework designed to identify the most statistically significant predictors of depressive symptoms, thereby providing precise intervention guidance for organizations such as **UNICEF**.

### 1.2 Data Description
The data set is from **Multiple Indicator Cluster Survey (MICS)**.

**Data Source:** This dataset is a simplified, cleaned subset focusing on the situation in Malawi in 2019–2020, integrating information at the child, mother, and family levels.

**Feature Dimensions:** It includes variables related to demographics, education, labor, family environment, and living conditions, allowing for a comprehensive assessment of factors related to child well-being.

**Target Variable Transformation:** The original survey question (FCF26) recorded the frequency with which children felt anxious or depressed. As required by the project, this variable was simplified to a binary outcome, designed to distinguish between "no signs of depression" and "any degree of depressive tendency."


### 1.3 Methodological Approach

To extract the most explanatory model from numerous variables, we implemented the following multi-stage analysis process:

#### **First stage：Feature Selection**

1. **Numerical and ordered variables**：Using **Mutual Information** can capture complex nonlinear relationships between variables.

2. **Categorical Variables (Nominal):** A **Chi-square test($\chi^2$ Test)** was used. This ensures that the qualitative factors entering the model have statistically significant distributional differences from the target variable.

#### **Second stage: (Baseline Modeling)**

Build **Logistic Regression** as baseline model.

* **Formula**：
$$\ln\left(\frac{P}{1-P}\right) = \beta_0 + \beta_1X_1 + \beta_2X_2 + \dots + \beta_kX_k$$



### 1.4 Preliminary Conclusions


# Exploratory Data Analysis and Feature Engineering

*Include a detailed discussion of the data with a particular emphasis on the features of the data that are relevant for the subsequent modeling. Including visualizations of the data is strongly encouraged - all code and plots must also be described in the write up. Think carefully about whether each plot needs to be included in your final draft and the appropriate type of plot and summary for each variable type - your report should include figures but they should be as focused and impactful as possible.*

*You should also split your data into training and testing sets, ideally before you look to much into the features and relationships with the target.*

*Additionally, this section should also motivate and describe any preprocessing / feature engineering of the data. Pipelines should be used and feature engineering steps that are be performed as part of an sklearn pipeline can be mentioned here but should be implemented in the following section.*

*All code and figures should be accompanied by text that provides an overview / context to what is being done or presented.*

In [12]:
# 2. 复制一份分析用数据，避免直接改原始数据
df = df_raw.copy()
# 3. 先看整体数据基本情况
print("Overall dataset shape:", df.shape)
print("\nFirst 10 column names:")
print(df.columns[:10].tolist())
print("\nData types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())
child_background_vars = ['CB3','CB4','CB5A','CB5B','CB7','CB11','HH6','HH7','HL4','ethnicity','wscore']
child_labour_vars = ['CL2','CL3','CL12','CL13']
child_discipline_vars = ['FCD2A','FCD2B','FCD2C','FCD2D','FCD2E','FCD2F','FCD2G','FCD2H','FCD2I','FCD2J','FCD2K','FCD5']
child_functioning_vars = ['FCF26']
mother_background_vars = ['WB4', 'WB5', 'WB6A', 'WB6B', 'WB14']
domestic_violence_vars = ['DV1A', 'DV1B', 'DV1C', 'DV1D', 'DV1E']
victimization_vars = ['VT1', 'VT9', 'VT20', 'VT21', 'VT22A', 'VT22B', 'VT22C', 'VT22D', 'VT22E', 'VT22F', 'VT22X']
marriage_union_vars = ['MSTATUS', 'MA2', 'MA3']
adult_functioning_vars = ['disability','AF10','AF11','AF12']
tobacco_alcohol_vars = ['TA1','TA14']
life_satisfaction_vars = ['LS1','LS2','LS3','LS4']
fertility_vars = ['CSURV','CDEAD']
household_characteristics_vars = ['HC4','HC5','HC8','HC11','HC12','HC13','HC14','HC15', 'HC17','HC19']
insecticide_treated_net_vars = ['TN1']
Water_sanitation_vars = [ 'WS1', 'WS3', 'WS4', 'WS7', 'WS11', 'WS14', 'WS15' ]
handwashing_vars = ['HW5']
selected_vars = (
    handwashing_vars)
# 只保留你负责的变量
df_maternal = df[selected_vars].copy()
print("Maternal subset shape:", df_maternal.shape)
print("\nSelected columns:")
print(df_maternal.columns.tolist())
print("\nData types of selected variables:")
print(df_maternal.dtypes)
print("\nFirst 5 rows of selected variables:")
print(df_maternal.head())

# 每列缺失情况
missing_summary = pd.DataFrame({
    'missing_count': df_maternal.isna().sum(),
    'missing_percent': df_maternal.isna().mean() * 100,
    'dtype': df_maternal.dtypes
}).sort_values(by='missing_percent', ascending=False)
print(missing_summary)

# 每列取值有哪些
for col in df_maternal.columns:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(f"dtype: {df_maternal[col].dtype}")
    print(f"Number of unique values (including NaN): {df_maternal[col].nunique(dropna=False)}")
    print(df_maternal[col].value_counts(dropna=False).sort_index().head(20))

# # 1) WB5 与 WB6A 缺失的交叉表
# ct_wb6a = pd.crosstab(df['WB5'], df['WB6A'].isna(), margins=True)
# print("WB5 vs WB6A missing:")
# print(ct_wb6a)
# # 2) WB5 与 WB6B 缺失的交叉表
# ct_wb6b = pd.crosstab(df['WB5'], df['WB6B'].isna(), margins=True)
# print("\nWB5 vs WB6B missing:")
# print(ct_wb6b)
## WB4存在NaN
## WB5: 是否上过学
df["WB5"] = df["WB5"].replace("NO RESPONSE", np.nan)
## WB6A:最高教育层级，处理结构性缺失
df.loc[df["WB5"] == "NO", "WB6A"] = "NoSchool"
## WB6B:最高教育年数。处理结构性缺失
df["WB6B"] = df["WB6B"].replace("DK", np.nan)
df.loc[df["WB5"] == "NO", "WB6B"] = '0'
## WB14:NO RESPONSE就是真实的NaN,因为所有人都要问这个问题
df["WB14"] = df["WB14"].replace("NO RESPONSE", np.nan)
cols_to_check = ['WB4',"WB5", "WB6A", "WB6B", "WB14"]
## DV1A-E 直接把NaN与NO RESPONCE改成NaN
dv_cols = ["DV1A", "DV1B", "DV1C", "DV1D", "DV1E"]
for col in dv_cols:
    df[col] = df[col].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan
    })
##  Victimization
vic_cols=['VT1', 'VT9', 'VT20', 'VT21', 'VT22A', 'VT22B', 'VT22C', 'VT22D', 'VT22E', 'VT22F', 'VT22X']
for col in vic_cols:
    df[col] = df[col].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan
    })
## Marriage/Union
# # 1) Marriage 与 MA2 缺失的交叉表
# ct_ma2 = pd.crosstab(df['MSTATUS'], df['MA2'].isna(), margins=True)
# print("MSTATUS vs MA2:")
# print(ct_ma2)
# # 2) WB5 与 WB6B 缺失的交叉表
# ct_ma3 = pd.crosstab(df['MSTATUS'], df['MA3'].isna(), margins=True)
# print("MSTATUS vs MA3:")
# print(ct_ma3)

df["MSTATUS"] = df["MSTATUS"].replace("9.0", 'Currently married/in union')
df['MA3']=df['MA3'].replace('NO RESPONSE',np.nan)
df.loc[df["MSTATUS"].isin(["Never married/in union", "Formerly married/in union"]), "MA3"] = "NotApplicable"
#结构性缺失
df['MA2']=df['MA2'].replace(['DK','NO RESPONSE'],np.nan)
df["MA2_status"] = "observed"
#在当前非婚姻状态下（从未接过婚/离过婚），MA2不适用
mask_structural = df["MSTATUS"].isin([
    "Never married/in union",
    "Formerly married/in union"
])
#非结构性缺失:当前婚姻状态下，本该有值但缺失
mask_genuine_missing = (
    (df["MSTATUS"] == "Currently married/in union") &
    (df["MA2"].isna())
)
df.loc[mask_structural, "MA2_status"] = "structural_missing"
df.loc[mask_genuine_missing, "MA2_status"] = "genuine_missing"
mar_var=['MSTATUS','MA2','MA2_status','MA3']

# Adultfunctioning 无需处理
af_var=['disability','AF10','AF11']
# Tobacco and alcoholuse
ta_var=['TA1','TA14']
for col in ta_var:
    df[col] = df[col].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan
    })
# Lifesatisfaction
ls_var=['LS1','LS2','LS3','LS4']
for col in ls_var:
    df[col] = df[col].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan
    })
# Fertility 无需处理
fer_var=['CSURV','CDEAD']

## Child-level variables
# Child_background
# # 1) CB4 与 CB5A 缺失的交叉表
# ct_cb5a = pd.crosstab(df['CB4'], df['CB5A'].isna(), margins=True)
# print("CB4 vs CB6A missing:")
# print(ct_cb5a)
# # 2) CB4 与 CB6B 缺失的交叉表
# ct_cb5b = pd.crosstab(df['CB4'], df['CB5B'].isna(), margins=True)
# print("\nCB4 vs CB6B missing:")
# print(ct_cb5b)
# # 3) CB4 与 CB7 缺失的交叉表
# ct_cb7 = pd.crosstab(df['CB4'], df['CB7'].isna(), margins=True)
# print("\nCB4 vs CB7 missing:")
# print(ct_cb7)
## CB4: 是否上过学
df["CB4"] = df["CB4"].replace("NO RESPONSE", np.nan)
## CB5A:最高教育层级，处理结构性缺失
df['CB5A']=df['CB5A'].replace("NO RESPONSE", np.nan)
df.loc[df["CB4"] == "NO", "CB5A"] = "NoSchool"
## CB5B:最高教育年数。处理结构性缺失
df.loc[df["CB4"] == "NO", "CB5B"] = '0'
## CB7:目前是否正在上学
df.loc[df["CB4"] == "NO", "CB7"] = 'NO'
## CB11
df["CB11"] = df["CB11"].replace("NO RESPONSE", np.nan)
age_child_var = ['CB3',"CB4", "CB5A", "CB5B", "CB7",'CB11']
# region 无需进行数据处理
clregion_var=['HH6','HH7','HL4','ethnicity','wscore']

# ## Childlabour
# # 1) CL2 与 CL3 缺失的交叉表
# ct_cl3 = pd.crosstab(df['CL2'], df['CL3'].isna(), margins=True)
# print("CL2 vs CL3 missing:")
# print(ct_cl3)
# # 2) CL12 与 CL13 缺失的交叉表
# ct_cl13 = pd.crosstab(df['CL12'], df['CL13'].isna(), margins=True)
# print("\nCL12 vs CL13 missing:")
# print(ct_cl13)
## 已知原本CL3与CL13就有00表示有劳动，但劳动少于1小时，
# 所以无法直接将CL2与CL12='FALSE'后的值设置为0
# 所以设立指示变量
df["CL3_status"] = "observed"
mask_structuralcl3 = df["CL2"] == False
mask_genuinecl3 = (df["CL2"] == True) & (df["CL3"].isna())
df.loc[mask_structuralcl3, "CL3_status"] = "structural_missing"
df.loc[mask_genuinecl3, "CL3_status"] = "genuine_missing"
df["CL13_status"] = "observed"
mask_structuralcl13 = df["CL12"] == False
mask_genuinecl13 = (df["CL12"] == True) & (df["CL13"].isna())
df.loc[mask_structuralcl13, "CL13_status"] = "structural_missing"
df.loc[mask_genuinecl13, "CL13_status"] = "genuine_missing"
cl_var=['CL2','CL3','CL12','CL13','CL3_status','CL13_status']
# Childdiscipline
cd_var=['FCD2A','FCD2B','FCD2C','FCD2D','FCD2E','FCD2F','FCD2G','FCD2H','FCD2I','FCD2J','FCD2K','FCD5']
for col in cd_var:
    df[col] = df[col].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan,
        "DK / NO OPINION":np.nan
    })
# Childfunctioning
df["FCF26"] = df["FCF26"].replace("NO RESPONSE", np.nan)
fcf26_map = {
    "NEVER": 'No',
    "A FEW TIMES A YEAR": 'YES',
    "MONTHLY": 'YES',
    "WEEKLY": 'YES',
    "DAILY": 'YES'
}
df['FCF26']=df['FCF26'].map(fcf26_map)

## Household variables
# Household characteristics(无结构缺失影响)
hc_var=['HC4','HC5','HC8','HC11','HC12','HC13','HC14','HC15', 'HC17','HC19']
for col in hc_var:
    df[col] = df[col].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan,
        "DK / NO OPINION":np.nan
    })
#  Insecticide treated nets
df['TN1']= df['TN1'].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan,
        "DK / NO OPINION":np.nan
    })
## Water and sanitation  ##如果要用最先用WS1与WS11，后面太复杂了
# # 1) WS1 与 WS3 缺失的交叉表
# ct_ws3 = pd.crosstab(df['WS1'], df['WS3'].isna(), margins=True)
# print("WS1 vs WS3:")
# print(ct_ws3)
# # 2) WS1 与 WS4 缺失的交叉表
# ct_ws4 = pd.crosstab(df['WS1'], df['WS4'].isna(), margins=True)
# print("WS1 vs WS4:")
# print(ct_ws4)
# # 3)WS11与WS14缺失的交叉表
# ct_ws14 = pd.crosstab(df['WS11'], df['WS14'].isna(), margins=True)
# print("WS11 vs WS14:")
# print(ct_ws14)
# # 4）WS11与WS15缺失的交叉表
# ct_ws15 = pd.crosstab(df['WS11'], df['WS15'].isna(), margins=True)
# print("WS11 vs WS15:")
# print(ct_ws15)
df[['WS1','WS11']]= df[['WS1','WS11']].replace({
        "DK": np.nan,
        "NO RESPONSE": np.nan,
        "DK / NO OPINION":np.nan
    })
## Handwashing
df['HW5']=df['HW5'].replace({"NO RESPONSE": np.nan})
# df.to_csv("unicef_malawi_cleaned_full.csv", index=False)
# for col in ['HW5']:
#     print(f"\n{'='*50}")
#     print(f"Column: {col}")
#     print(f"dtype: {df[col].dtype}")
#     print(f"Number of unique values (including NaN): {df[col].nunique(dropna=False)}")
#     print(df[col].value_counts(dropna=False).sort_index().head(20))

Overall dataset shape: (13162, 87)

First 10 column names:
['HH1', 'HH2', 'LN', 'FS4', 'CB3', 'CB4', 'CB5A', 'CB5B', 'CB7', 'CB11']

Data types:
HH1     float64
HH2     float64
LN      float64
FS4     float64
CB3     float64
         ...   
WS7         str
WS11        str
WS14        str
WS15        str
HW5         str
Length: 87, dtype: object

First 5 rows:
   HH1   HH2   LN  FS4   CB3  CB4     CB5A           CB5B  CB7 CB11  ...  \
0  1.0   2.0  2.0  2.0  14.0  YES  PRIMARY  CLASS/GRADE 6  YES   NO  ...   
1  1.0   3.0  1.0  1.0   5.0  YES      ECE            NaN  YES   NO  ...   
2  1.0   4.0  2.0  2.0  16.0  YES  PRIMARY  CLASS/GRADE 7  YES   NO  ...   
3  1.0   8.0  2.0  2.0  13.0  YES  PRIMARY  CLASS/GRADE 7  YES   NO  ...   
4  1.0  10.0  2.0  2.0  14.0  YES  PRIMARY  CLASS/GRADE 6  YES   NO  ...   

   HC19  TN1                                  WS1        WS3   WS4  \
0    NO   NO  PIPED WATER: PUBLIC TAP / STANDPIPE  ELSEWHERE   5.0   
1    NO  YES                 TUBE WELL / 

In [ ]:
# =========================================================
# 0. 基础设置
# =========================================================

RANDOM_STATE = 42
TEST_SIZE = 0.2

# 如果你的目标变量是 child_depression 或 y，请确认这里一致
target_col = "FCF26"

# =========================================================
# 1. 读取数据
# =========================================================
print("原始数据 shape:", df.shape)

# =========================================================
# 2. 删除 FCF26 缺失值
# =========================================================

df = df.dropna(subset=["FCF26"]).copy()

print("删除FCF26缺失后 shape:", df.shape)

# =========================================================
# 3. train / test split（分层抽样）
# =========================================================

train_df_full, test_df_full = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df[target_col],   # 防止类别比例变化
    random_state=RANDOM_STATE
)

print("\nTrain shape:", train_df_full.shape)
print("Test shape :", test_df_full.shape)

# =========================================================
# 4. 重置索引（推荐）
# =========================================================

train_df_full = train_df_full.reset_index(drop=True)
test_df_full = test_df_full.reset_index(drop=True)

print("\n数据划分完成 ✅")

原始数据 shape: (13036, 90)
删除FCF26缺失后 shape: (13036, 90)

Train shape: (10428, 90)
Test shape : (2608, 90)

数据划分完成 ✅


In [ ]:
# =========================================================
# Cell 2: 变量筛选（可直接完整替代）
# 前提：
# - train_df_full, test_df_full 已存在
# - target_col 已存在
# =========================================================
# =========================================================
# 0. 参数设置
# =========================================================
RANDOM_STATE = 42

missing_thresh = 0.30      # 只在 train 上判断缺失率
mi_thresh = 0.001          # 数值/有序变量 MI 阈值
chi2_p_thresh = 0.05       # 分类变量卡方检验阈值

# =========================================================
# 1. 基本检查
# =========================================================
train_df = train_df_full.copy()
test_df = test_df_full.copy()

assert target_col in train_df.columns, f"{target_col} 不在 train_df 中"
assert target_col in test_df.columns, f"{target_col} 不在 test_df 中"

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

X_train = train_df.drop(columns=[target_col]).copy()
X_test = test_df.drop(columns=[target_col]).copy()

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("target_col   :", target_col)

# =========================================================
# 2. 先按 train 缺失率做第一轮筛选
# =========================================================
missing_rate = X_train.isna().mean().sort_values(ascending=False)

keep_by_missing = missing_rate[missing_rate <= missing_thresh].index.tolist()
drop_by_missing = missing_rate[missing_rate > missing_thresh].index.tolist()

X_train = X_train[keep_by_missing].copy()
X_test = X_test[keep_by_missing].copy()

print("\n=== 缺失率筛选 ===")
print("保留变量数:", len(keep_by_missing))
print("删除变量数:", len(drop_by_missing))
if len(drop_by_missing) > 0:
    print("因缺失率过高删除的变量:")
    print(drop_by_missing)

# =========================================================
# 3. 定义变量类型
#    注意：这里把“真正有顺序”的变量单独列出来做手动映射
# =========================================================
numeric_vars_all = [
    "HH1", "HH2", "LN", "FS4",
    "CB3", "CL3", "CL13", "wscore", "WB4",
    "MA2", "LS2", "CSURV", "CDEAD"
]

# 手动定义有序变量映射
ordinal_maps = {
    # 教育水平
    "CB5A": {
        "NoSchool": 0,
        "ECE": 1,
        "PRIMARY": 2,
        "LOWER SECONDARY": 3,
        "UPPER SECONDARY": 4,
        "HIGHER": 5
    },
    "WB6A": {
        "NoSchool": 0,
        "ECE": 1,
        "PRIMARY": 2,
        "LOWER SECONDARY": 3,
        "UPPER SECONDARY": 4,
        "VOCATIONAL TRAINING": 5,
        "HIGHER": 6
    },

    # 年级
    "CB5B": {
        "0": 0,
        "CLASS/YEAR/GRADE 1": 1,
        "CLASS/YEAR/GRADE 2": 2,
        "CLASS/YEAR/GRADE 3": 3,
        "CLASS/YEAR/GRADE 4": 4,
        "CLASS/YEAR/GRADE 5": 5,
        "CLASS/GRADE 6": 6,
        "CLASS/GRADE 7": 7,
        "CLASS/GRADE 8": 8
    },
    "WB6B": {
        "0": 0,
        "CLASS/YEAR/GRADE 1": 1,
        "CLASS/YEAR/GRADE 2": 2,
        "CLASS/YEAR/GRADE 3": 3,
        "CLASS/YEAR/GRADE 4": 4,
        "CLASS/YEAR/GRADE 5": 5,
        "CLASS/GRADE 6": 6,
        "CLASS/GRADE 7": 7,
        "CLASS/GRADE 8": 8
    },

    # 阅读能力
    "WB14": {
        "CANNOT READ AT ALL": 0,
        "NO SENTENCE IN REQUIRED LANGUAGE / BRAILLE": 1,
        "ABLE TO READ ONLY PARTS OF SENTENCE": 2,
        "ABLE TO READ WHOLE SENTENCE": 3
    },

    # 功能困难程度
    "AF10": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2
    },
    "AF11": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2
    },
    "AF12": {
        "NO DIFFICULTY": 0,
        "SOME DIFFICULTY": 1,
        "A LOT OF DIFFICULTY": 2,
        "NO RESPONSE": np.nan
    },

    # 幸福感
    "LS1": {
        "VERY UNHAPPY": 0,
        "SOMEWHAT UNHAPPY": 1,
        "NEITHER HAPPY NOR UNHAPPY": 2,
        "SOMEWHAT HAPPY": 3,
        "VERY HAPPY": 4
    },

    # 生活变化
    "LS3": {
        "WORSENED": 0,
        "MORE OR LESS THE SAME": 1,
        "IMPROVED": 2
    },
    "LS4": {
        "WORSE": 0,
        "MORE OR LESS THE SAME": 1,
        "BETTER": 2
    },

    # 夜间安全感
    "VT20": {
        "VERY UNSAFE": 0,
        "UNSAFE": 1,
        "SAFE": 2,
        "VERY SAFE": 3,
        "NEVER WALK ALONE AFTER DARK": np.nan
    },
    "VT21": {
        "VERY UNSAFE": 0,
        "UNSAFE": 1,
        "SAFE": 2,
        "VERY SAFE": 3,
        "NEVER WALK ALONE AFTER DARK": np.nan
    },

    # 婚姻状态
    "MSTATUS": {
        "Never married/in union": 0,
        "Formerly married/in union": 1,
        "Currently married/in union": 2
    },

    # 配偶是否有其他妻子
    "MA3": {
        "NO": 0,
        "YES": 1,
        "NotApplicable": np.nan
    }
}

# 只保留当前数据中实际存在的数值变量 / 有序变量
numeric_vars = [c for c in numeric_vars_all if c in X_train.columns]
ordinal_vars = [c for c in ordinal_maps.keys() if c in X_train.columns]

# 其余都作为分类变量
categorical_vars = [
    c for c in X_train.columns
    if c not in numeric_vars + ordinal_vars
]

print("\n=== 变量类型划分 ===")
print("数值变量数:", len(numeric_vars))
print("有序变量数:", len(ordinal_vars))
print("分类变量数:", len(categorical_vars))

# =========================================================
# 4. 手动将有序变量映射为数字
# =========================================================
for col in ordinal_vars:
    mapping = ordinal_maps[col]
    X_train[col] = X_train[col].map(mapping)
    X_test[col] = X_test[col].map(mapping)

# 一些本应为数值的列可能是 object，强制转数值
for col in numeric_vars:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_test[col] = pd.to_numeric(X_test[col], errors="coerce")

# =========================================================
# 5. 数值变量 + 有序变量：MI 筛选
# =========================================================
selected_num_ord = []
num_ord_vars = numeric_vars + ordinal_vars

if len(num_ord_vars) > 0:
    X_train_num_ord = X_train[num_ord_vars].copy()

    # 再保险：全部转数值
    for col in X_train_num_ord.columns:
        X_train_num_ord[col] = pd.to_numeric(X_train_num_ord[col], errors="coerce")

    # 删除全空列，避免 SimpleImputer 输出列数不一致
    all_nan_cols = X_train_num_ord.columns[X_train_num_ord.isna().all()].tolist()
    if len(all_nan_cols) > 0:
        print("\n以下 num/ord 变量在 train 中全为 NaN，已删除：")
        print(all_nan_cols)
        X_train_num_ord = X_train_num_ord.drop(columns=all_nan_cols)

    if X_train_num_ord.shape[1] > 0:
        mi_imputer = SimpleImputer(strategy="median")
        X_train_num_ord_imp = pd.DataFrame(
            mi_imputer.fit_transform(X_train_num_ord),
            columns=X_train_num_ord.columns,
            index=X_train_num_ord.index
        )

        mi_scores = mutual_info_classif(
            X_train_num_ord_imp,
            y_train.loc[X_train_num_ord_imp.index],
            random_state=RANDOM_STATE
        )

        mi_df = pd.DataFrame({
            "variable": X_train_num_ord_imp.columns,
            "mi_score": mi_scores
        }).sort_values("mi_score", ascending=False)

        selected_num_ord = mi_df.loc[
            mi_df["mi_score"] >= mi_thresh, "variable"
        ].tolist()
    else:
        mi_df = pd.DataFrame(columns=["variable", "mi_score"])
else:
    mi_df = pd.DataFrame(columns=["variable", "mi_score"])

print("\n=== 数值/有序变量 MI 结果 ===")
print(mi_df)
print("保留的数值/有序变量:", selected_num_ord)

# =========================================================
# 6. 分类变量：卡方检验筛选
# =========================================================
selected_cat = []
chi2_results = []

for col in categorical_vars:
    tmp = pd.DataFrame({
        col: X_train[col],
        target_col: y_train
    }).dropna()

    # 类别数过少或目标只有一个类别时跳过
    if tmp[col].nunique() < 2 or tmp[target_col].nunique() < 2:
        continue

    contingency = pd.crosstab(tmp[col], tmp[target_col])

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        continue

    try:
        chi2, p_value, dof, expected = chi2_contingency(contingency)
        chi2_results.append({
            "variable": col,
            "chi2_p_value": p_value
        })

        if p_value < chi2_p_thresh:
            selected_cat.append(col)

    except Exception as e:
        print(f"卡方检验跳过 {col}: {e}")

chi2_df = pd.DataFrame(chi2_results).sort_values("chi2_p_value", ascending=True)

print("\n=== 分类变量卡方检验结果 ===")
print(chi2_df)
print("保留的分类变量:", selected_cat)

# =========================================================
# 7. 合并筛选结果
# =========================================================
selected_features_first_round = selected_num_ord + selected_cat

print("\n=== 第一轮筛选后变量 ===")
print("变量总数:", len(selected_features_first_round))
print(selected_features_first_round)

# =========================================================
# 8. 生成后续可直接使用的数据
# =========================================================
train_screened = pd.concat(
    [X_train[selected_features_first_round], y_train],
    axis=1
).copy()

test_screened = pd.concat(
    [X_test[selected_features_first_round], y_test],
    axis=1
).copy()

print("\n=== 筛选后数据形状 ===")
print("train_screened shape:", train_screened.shape)
print("test_screened shape :", test_screened.shape)

# 可选：查看最终保留变量分类
selected_numeric_vars = [c for c in selected_features_first_round if c in numeric_vars]
selected_ordinal_vars = [c for c in selected_features_first_round if c in ordinal_vars]
selected_categorical_vars = [c for c in selected_features_first_round if c in categorical_vars]

print("\n=== 最终保留变量类型 ===")
print("数值变量:", selected_numeric_vars)
print("有序变量:", selected_ordinal_vars)
print("分类变量:", selected_categorical_vars)

X_train shape: (10428, 89)
X_test shape : (2608, 89)
target_col   : FCF26

=== 缺失率筛选 ===
保留变量数: 87
删除变量数: 2
因缺失率过高删除的变量:
['CL3', 'FCD5']

=== 变量类型划分 ===
数值变量数: 12
有序变量数: 15
分类变量数: 60

=== 数值/有序变量 MI 结果 ===
   variable  mi_score
0       HH1  0.031502
25  MSTATUS  0.015554
3       FS4  0.012010
22      LS4  0.010794
20      LS1  0.010110
8       MA2  0.008371
24     VT21  0.008095
23     VT20  0.007510
16     WB14  0.006472
10    CSURV  0.004869
12     CB5A  0.004414
13     WB6A  0.004275
5      CL13  0.002950
7       WB4  0.002930
1       HH2  0.002850
15     WB6B  0.001868
9       LS2  0.001809
18     AF11  0.000771
19     AF12  0.000540
4       CB3  0.000000
6    wscore  0.000000
2        LN  0.000000
11    CDEAD  0.000000
17     AF10  0.000000
14     CB5B  0.000000
21      LS3  0.000000
26      MA3  0.000000
保留的数值/有序变量: ['HH1', 'MSTATUS', 'FS4', 'LS4', 'LS1', 'MA2', 'VT21', 'VT20', 'WB14', 'CSURV', 'CB5A', 'WB6A', 'CL13', 'WB4', 'HH2', 'WB6B', 'LS2']

=== 分类变量卡方检验结果 ===
       variab

In [20]:
# =========================================================
# Cell 3: 在第一轮筛选结果基础上做手动调整
# 重点：
# - 自动筛选结果来自上一个 cell 的 selected_features_first_round
# - 手动添加变量从 train_df_full / test_df_full 中找
# - 输出 train_final / test_final，供后续模型 cell 直接使用
# =========================================================
# =========================================================
# 0. 承接前面 cell 的对象
# 前提：
# - train_df_full
# - test_df_full
# - train_df
# - test_df
# - selected_features_first_round
# - target_col
# 都已经在前面的 cell 中生成
# =========================================================

print("train_df_full shape:", train_df_full.shape)
print("test_df_full shape :", test_df_full.shape)
print("train_df shape     :", train_df.shape)
print("test_df shape      :", test_df.shape)
print("初步筛选后变量数:", len(selected_features_first_round))
print(selected_features_first_round)

# =========================================================
# 1. 安全检查
# =========================================================
print("\n=== 数据检查 ===")
print("target_col:", target_col)
print("target 是否在 train_df_full 中:", target_col in train_df_full.columns)
print("target 是否在 test_df_full 中 :", target_col in test_df_full.columns)
print("target 是否在 train_df 中     :", target_col in train_df.columns)
print("target 是否在 test_df 中      :", target_col in test_df.columns)

assert target_col in train_df_full.columns, f"{target_col} 不在 train_df_full 中"
assert target_col in test_df_full.columns, f"{target_col} 不在 test_df_full 中"

# =========================================================
# 2. 手动删除变量
# =========================================================
manual_drop_vars = [
    "VT22A", "VT22B", "VT22C", "VT22D", "VT22E", "VT22F", "VT22X"
]

# =========================================================
# 3. 手动添加变量（注意：从 train_df_full / test_df_full 里检查）
# =========================================================
theory_add_vars = [
    "CB3",
    "wscore",
    "WB5",
    "CB4"
]

# =========================================================
# 4. 强制保留变量
# =========================================================
structural_keep_vars = [
    "CL3_status",
    "CL13_status",
    "MA2_status"
]

# =========================================================
# 5. 过滤变量
#    - 删除变量：只能删掉“第一轮已选中”的变量
#    - 添加变量：必须真实存在于 train_df_full 和 test_df_full
# =========================================================
manual_drop_vars = [v for v in manual_drop_vars if v in selected_features_first_round]

theory_add_vars = [
    v for v in theory_add_vars
    if (v in train_df_full.columns)
    and (v in test_df_full.columns)
    and (v != target_col)
]

structural_keep_vars = [
    v for v in structural_keep_vars
    if (v in train_df_full.columns)
    and (v in test_df_full.columns)
    and (v != target_col)
]

manual_add_vars = sorted(set(theory_add_vars + structural_keep_vars))

# =========================================================
# 6. 生成最终变量列表
# =========================================================
selected_features_final = [
    v for v in selected_features_first_round
    if v not in manual_drop_vars
]

for v in manual_add_vars:
    if v not in selected_features_final:
        selected_features_final.append(v)

# 再做一次保险：只保留 full train/test 都存在的列
selected_features_final = [
    v for v in selected_features_final
    if (v in train_df_full.columns) and (v in test_df_full.columns) and (v != target_col)
]

print("\n手动删除变量:")
print(manual_drop_vars)

print("\n手动添加变量（从 train_df_full / test_df_full 检查后）:")
print(manual_add_vars)

print("\n最终变量数:", len(selected_features_final))
print(selected_features_final)

# =========================================================
# 7. 用 full 数据生成最终 train / test
#    注意这里用的是 train_df_full / test_df_full
#    这样手动添加的变量一定能接进去
# =========================================================
train_df = train_df_full[selected_features_final + [target_col]].copy()
test_df = test_df_full[selected_features_final + [target_col]].copy()

print("\ntrain_df shape:", train_df.shape)
print("test_df shape :", test_df.shape)

# =========================================================
# 8. 输出变量调整记录
# =========================================================
adjustment_records = []

all_vars_seen = sorted(set(selected_features_first_round) | set(manual_add_vars))

for v in all_vars_seen:
    adjustment_records.append({
        "variable": v,
        "auto_selected": "Yes" if v in selected_features_first_round else "No",
        "manually_dropped": "Yes" if v in manual_drop_vars else "No",
        "manually_added": "Yes" if (v in manual_add_vars and v not in selected_features_first_round) else "No",
        "final_selected": "Yes" if v in selected_features_final else "No"
    })

adjustment_summary_df = pd.DataFrame(adjustment_records).sort_values(
    by=["final_selected", "auto_selected", "variable"],
    ascending=[False, False, True]
)

print("\n变量调整记录表：")
display(adjustment_summary_df)

train_df.isna().sum().sort_values(ascending=False).head(20)


train_df_full shape: (10428, 90)
test_df_full shape : (2608, 90)
train_df shape     : (10428, 90)
test_df shape      : (2608, 90)
初步筛选后变量数: 62
['HH1', 'MSTATUS', 'FS4', 'LS4', 'LS1', 'MA2', 'VT21', 'VT20', 'WB14', 'CSURV', 'CB5A', 'WB6A', 'CL13', 'WB4', 'HH2', 'WB6B', 'LS2', 'HW5', 'FCD2A', 'FCD2I', 'FCD2G', 'FCD2B', 'FCD2C', 'FCD2J', 'FCD2D', 'FCD2E', 'FCD2H', 'WS4', 'WS15', 'WS14', 'DV1B', 'VT22C', 'VT22B', 'VT1', 'DV1C', 'VT9', 'VT22A', 'TA14', 'VT22E', 'VT22D', 'VT22X', 'VT22F', 'disability', 'WB5', 'HC19', 'HC11', 'HC13', 'HC8', 'HC12', 'HC14', 'WS11', 'HH7', 'ethnicity', 'HC5', 'CL2', 'CL12', 'HC4', 'WS1', 'WS7', 'MA2_status', 'CL3_status', 'CL13_status']

=== 数据检查 ===
target_col: FCF26
target 是否在 train_df_full 中: True
target 是否在 test_df_full 中 : True
target 是否在 train_df 中     : True
target 是否在 test_df 中      : True

手动删除变量:
['VT22A', 'VT22B', 'VT22C', 'VT22D', 'VT22E', 'VT22F', 'VT22X']

手动添加变量（从 train_df_full / test_df_full 检查后）:
['CB3', 'CB4', 'CL13_status', 'CL3_status', 'MA2

,variable,auto_selected,manually_dropped,manually_added,final_selected
2,CB5A,Yes,No,No,Yes
3,CL12,Yes,No,No,Yes
4,CL13,Yes,No,No,Yes
5,CL13_status,Yes,No,No,Yes
6,CL2,Yes,No,No,Yes
...,...,...,...,...,...
45,VT22C,Yes,Yes,No,No
46,VT22D,Yes,Yes,No,No
47,VT22E,Yes,Yes,No,No
48,VT22F,Yes,Yes,No,No


HW5      2753
MA2      2489
WB14     2292
CL13     1573
FCD2A    1479
FCD2I    1454
FCD2G    1449
FCD2B    1448
FCD2C    1448
FCD2J    1446
FCD2E    1445
FCD2D    1445
FCD2H    1444
WS4      1402
WS15      762
WS14      762
LS4       741
LS2       276
DV1B      268
VT1       254
dtype: int64

In [38]:
random_state = 42

# 使用手动筛选后的变量
baseline_vars = selected_features_final

X_train = train_df[baseline_vars].copy()
y_train = train_df[target_col].copy()

X_test = test_df[baseline_vars].copy()
y_test = test_df[target_col].copy()

# =========================================================
# 0.1 目标变量编码：No -> 0, YES -> 1
# =========================================================
label_map = {
    "No": 0,
    "YES": 1
}

y_train = train_df[target_col].map(label_map)
y_test = test_df[target_col].map(label_map)
print("进入 baseline 的变量数:", len(baseline_vars))
print(baseline_vars)

# =========================================================
# 1. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]
X_train[categorical_cols] = X_train[categorical_cols].astype(str)  #这里是我加的，因为我后面baseline model我没跑出来
X_test[categorical_cols] = X_test[categorical_cols].astype(str)  #加上这两行以后能跑了，但是模型好像变差了

# =========================================================
# 2. 预处理 pipeline
# =========================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

进入 baseline 的变量数: 58
['HH1', 'MSTATUS', 'FS4', 'LS4', 'LS1', 'MA2', 'VT21', 'VT20', 'WB14', 'CSURV', 'CB5A', 'WB6A', 'CL13', 'WB4', 'HH2', 'WB6B', 'LS2', 'HW5', 'FCD2A', 'FCD2I', 'FCD2G', 'FCD2B', 'FCD2C', 'FCD2J', 'FCD2D', 'FCD2E', 'FCD2H', 'WS4', 'WS15', 'WS14', 'DV1B', 'VT1', 'DV1C', 'VT9', 'TA14', 'disability', 'WB5', 'HC19', 'HC11', 'HC13', 'HC8', 'HC12', 'HC14', 'WS11', 'HH7', 'ethnicity', 'HC5', 'CL2', 'CL12', 'HC4', 'WS1', 'WS7', 'MA2_status', 'CL3_status', 'CL13_status', 'CB3', 'CB4', 'wscore']


# Model Fitting and Tuning

*In this section you should detail and motivate your choice of model and describe the process used to refine, tune, and fit that model. You are encouraged to explore different models but you should NOT include a detailed narrative or code of all of these attempts. At most this section should very briefly mention the methods explored and why they were rejected - most of your effort should go into describing the final model you are using and your process for tuning and validating it.*

*This section should include the full implementation of your final model, including all necessary validation. As with figures, any included code must also be addressed in the text of the document.*

*You should also include a baseline model of your choice and provide a comparison of your model with the baseline model on the test data. You should briefly describe the baseline model considered.*

In [39]:
# =========================================================
# 3. Baseline Logistic Regression
# =========================================================

baseline_model = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=5000,
    random_state=random_state
)

baseline_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", baseline_model)
])

baseline_pipeline.fit(X_train, y_train)


# =========================================================
# 4. 评估函数
# =========================================================

def evaluate_model(model, X, y):

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }, y_pred, y_prob


baseline_train_metrics, baseline_train_pred, baseline_train_prob = evaluate_model(
    baseline_pipeline, X_train, y_train
)

baseline_test_metrics, baseline_test_pred, baseline_test_prob = evaluate_model(
    baseline_pipeline, X_test, y_test
)


baseline_perf_df = pd.DataFrame(
    [baseline_train_metrics, baseline_test_metrics],
    index=["train", "test"]
)

print("\nBaseline表现：")
display(baseline_perf_df)

# =========================================================
# 5. 提取系数
# =========================================================

feature_names_after_encoding = baseline_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

baseline_coef_df = pd.DataFrame({
    "feature": feature_names_after_encoding,
    "coef": baseline_pipeline.named_steps["model"].coef_[0]
}).sort_values("coef", key=np.abs, ascending=False)

display(baseline_coef_df.head(20))

# =========================================================
# 6. 预测结果（供后续cell使用）
# =========================================================

baseline_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": baseline_train_pred,
    "y_prob": baseline_train_prob
})

baseline_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": baseline_test_pred,
    "y_prob": baseline_test_prob
})

/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(



Baseline表现：


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
train,0.63435,0.622572,0.642373,0.747896,0.691130,0.675939
test,0.59816,0.585137,0.612463,0.723196,0.663239,0.628307


,feature,coef
211,cat__WS4_13.0,-8.971721
246,cat__WS4_34.0,-7.665572
149,cat__CL13_45.0,6.532477
348,cat__HC5_ROOFING SHINGLES,-6.299190
229,cat__WS4_21.0,-5.829728
77,cat__MA2_72.0,-5.667532
213,cat__WS4_135.0,5.513106
78,cat__MA2_73.0,-5.472060
283,cat__WS15_NO RESPONSE,5.431962
85,cat__MA2_80.0,5.383138


In [41]:
Cs_grid = np.logspace(-2, 2, 30)

# 使用手动筛选变量
final_vars = selected_features_final
X_train = train_df[final_vars].copy()
y_train = train_df[target_col].map({"No": 0, "YES": 1}).copy()

X_test = test_df[final_vars].copy()
y_test = test_df[target_col].map({"No": 0, "YES": 1}).copy()

print("进入 Final(L1) 的变量数:", len(final_vars))

# =========================================================
# 1. 识别变量类型
# =========================================================

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [col for col in X_train.columns if col not in numeric_cols]
X_train[categorical_cols] = X_train[categorical_cols].astype(str)
X_test[categorical_cols] = X_test[categorical_cols].astype(str)

# =========================================================
# 3. Final：L1 LogisticRegressionCV
# =========================================================

final_model = LogisticRegressionCV(
    Cs=Cs_grid,
    cv=5,
    penalty="l1",
    solver="saga",
    scoring="roc_auc",
    max_iter=8000,
    random_state=random_state,
    n_jobs=-1,
    refit=True
)

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model)
])

final_pipeline.fit(X_train, y_train)

best_C_final = final_pipeline.named_steps["model"].C_[0]
print("\nFinal(L1)最优 C:", best_C_final)

# =========================================================
# 4. 评估函数
# =========================================================

def evaluate_model(model, X, y):

    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob)
    }, y_pred, y_prob


final_train_metrics, final_train_pred, final_train_prob = evaluate_model(
    final_pipeline, X_train, y_train
)

final_test_metrics, final_test_pred, final_test_prob = evaluate_model(
    final_pipeline, X_test, y_test
)


final_perf_df = pd.DataFrame(
    [final_train_metrics, final_test_metrics],
    index=["train", "test"]
)

print("\nFinal(L1)表现：")
display(final_perf_df)

# =========================================================
# 5. 提取 L1 非零变量
# =========================================================

feature_names_after_encoding = final_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

coefs = final_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature_after_encoding": feature_names_after_encoding,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs)
})

selected_coef_df = coef_df[coef_df["coefficient"] != 0].copy()

selected_coef_df = selected_coef_df.sort_values(
    "abs_coefficient",
    ascending=False
)


# 还原到原始变量层面

original_feature_names = X_train.columns.tolist()


def recover_original_variable(encoded_name):

    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "")

    elif encoded_name.startswith("cat__"):

        temp = encoded_name.replace("cat__", "")

        sorted_cols = sorted(
            original_feature_names,
            key=len,
            reverse=True
        )

        for col in sorted_cols:

            if temp == col or temp.startswith(col + "_"):
                return col

        return temp

    else:
        return encoded_name


selected_coef_df["original_variable"] = selected_coef_df[
    "feature_after_encoding"
].apply(recover_original_variable)


final_original_vars = selected_coef_df[
    "original_variable"
].drop_duplicates().tolist()


print("\nL1筛选后保留下来的变量数:", len(final_original_vars))

display(final_original_vars)


# =========================================================
# 6. 预测结果（供后续 cell 使用）
# =========================================================

final_train_pred_df = pd.DataFrame({
    "y_true": y_train,
    "y_pred": final_train_pred,
    "y_prob": final_train_prob
})

final_test_pred_df = pd.DataFrame({
    "y_true": y_test,
    "y_pred": final_test_pred,
    "y_prob": final_test_prob
})

进入 Final(L1) 的变量数: 58


/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1811: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' instead. Use l1_ratios=(0,) instead of penalty='l2'  and l1_ratios=(1,) instead of penalty='l1'.
  warnings.warn(
/usr/local/python/3.12.1/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=


Final(L1)最优 C: 0.09236708571873861

Final(L1)表现：


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
train,0.622746,0.608911,0.629084,0.756136,0.686783,0.653692
test,0.608512,0.593576,0.616667,0.751927,0.677613,0.636822



L1筛选后保留下来的变量数: 50


['WS7',
 'ethnicity',
 'VT20',
 'LS2',
 'WS4',
 'VT1',
 'WS14',
 'LS1',
 'FCD2H',
 'HH7',
 'VT21',
 'WS11',
 'MSTATUS',
 'LS4',
 'MA2',
 'HW5',
 'wscore',
 'HC8',
 'FCD2E',
 'WS1',
 'disability',
 'CL2',
 'CL3_status',
 'MA2_status',
 'WS15',
 'TA14',
 'FCD2B',
 'FCD2D',
 'DV1B',
 'WB6B',
 'WB4',
 'CL13_status',
 'CL12',
 'FCD2A',
 'CL13',
 'CB3',
 'WB6A',
 'WB14',
 'HC12',
 'VT9',
 'CB5A',
 'FCD2C',
 'HH1',
 'HC4',
 'FCD2J',
 'HH2',
 'HC13',
 'DV1C',
 'FS4',
 'CSURV']

# Interpretation, Discussion & Conclusions

*In this section you should provide a general overview of your final model, its performance, and reliability. You should discuss what the implications of your model are in terms of the included features, estimated parameters and relationships, predictive performance, and anything else you think is relevant.*

*This should be written with a target audience of a government official, who understands the issues associated with mental health but may only have university level mathematics (not necessarily postgraduate statistics or machine learning). Your goal should be to highlight to this audience how your model can useful. You should also discuss potential limitations or directions of future improvement of your model.*

*Finally, you should include recommendations on factors that may increase the risk of depression, which may be useful for the government officials and health care workers to improve their understanding of the condition, and potentially assit in the development of effective social and health policies and interventions.*

*Keep in mind that a negative result, i.e. a model that does not work well predictively, that is well explained and justified in terms of why it failed will likely receive higher marks than a model with strong predictive performance but with poor or incorrect explanations / justifications.*

# Generative AI statement

*Include a statement on how generative AI was used in the project and report.*

# References

*Include references if any*

In [ ]:
# Run the following to render to PDF
!jupyter nbconvert --to pdf project.ipynb